# Day 47: Regression: Predicting a Number

**Today's question:** can we predict a student's **final score** (a number from 0 to 100), and how do we say how
wrong we are, in points?

Yesterday's baseline, "everyone gets the average", missed by **8.8 points** on average. That is the number to beat.

Before running a cell marked 🤔, predict what it will show.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../../../datasets")
df = pd.read_csv(DATA_DIR / "students.csv")

## 1. Classification or regression?

| The target is… | Task | Example | We measure… |
|----------------|------|---------|-------------|
| a category (pass/fail, spam/not spam) | **classification** | `passed` | how often we are right |
| a number (score, price, temperature) | **regression** | `final_score` | how far off we are |

## 2. One feature: a line through the points

In [ ]:
plt.scatter(df["study_hours"], df["final_score"], alpha=0.5)
plt.xlabel("study hours per week")
plt.ylabel("final score")
plt.title("More study hours, higher scores (with lots of spread)")
plt.show()

The points drift upwards. If you had to draw **one straight line** through this cloud with a ruler, you would
try to keep it as close as possible to as many points as possible. `LinearRegression` does exactly that.

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["study_hours"]]
y = df["final_score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

line_model = LinearRegression()
line_model.fit(X_train, y_train)
print("slope (coef_):     ", line_model.coef_)
print("start (intercept_):", line_model.intercept_)

A line needs only two numbers:
- **`intercept_` ≈ 43.2**: the predicted score for a student who studies **0** hours (where the line starts).
- **`coef_` ≈ 1.49**: the slope. **Each extra study hour adds about 1.5 points** to the prediction.

So the model is just: `predicted score = 43.2 + 1.49 × study_hours`.

🤔 **Predict:** what score does the line give a student who studies 10 hours? Work it out, then run.

In [ ]:
print("by hand:", 43.15 + 1.4876 * 10)
print("model:  ", line_model.predict(pd.DataFrame([{"study_hours": 10}])))

Let's draw the line on top of the training points. `plt.plot(xs, ys)` (new) joins points with a line, so we
only need the predictions at 0 and 22 hours.

In [ ]:
ends = pd.DataFrame({"study_hours": [0, 22]})

plt.scatter(X_train["study_hours"], y_train, alpha=0.5, label="training students")
plt.plot(ends["study_hours"], line_model.predict(ends), color="red", label="learned line")
plt.xlabel("study hours per week")
plt.ylabel("final score")
plt.legend()
plt.show()

## 3. How wrong are we? Mean Absolute Error (MAE)

For each **test** student: miss = |actual − predicted|. Then take the average. Yesterday we called this the
"average miss"; its official name is the **mean absolute error (MAE)**. It is in the **same units as the target**:
points.

In [ ]:
predictions = line_model.predict(X_test)

comparison = pd.DataFrame({"actual": y_test, "predicted": predictions.round(1)})
comparison["miss"] = (comparison["actual"] - comparison["predicted"]).abs()
comparison.head()

In [ ]:
from sklearn.metrics import mean_absolute_error

print("MAE by hand:", comparison["miss"].mean())
print("MAE sklearn:", mean_absolute_error(y_test, predictions))

(The tiny difference comes from rounding the predictions in the table.)

🤔 **Predict:** the baseline missed by 8.8 points. Will the one-feature line beat it?

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

print("baseline MAE:", round(mean_absolute_error(y_test, baseline.predict(X_test)), 2))
print("line MAE:    ", round(mean_absolute_error(y_test, predictions), 2))

Yes: **7.3 points instead of 8.8**. Better, but a 7-point miss is still a lot. Study hours alone are not enough.

## 4. More features, and turning text into numbers

Models only understand numbers. `has_job` contains `"yes"`/`"no"`, so we make a 0/1 copy:
`df["has_job"] == "yes"` gives True/False, and `.astype(int)` (new) turns True → 1 and False → 0.

In [ ]:
df["has_job_num"] = (df["has_job"] == "yes").astype(int)
df[["has_job", "has_job_num"]].head()

In [ ]:
features = ["study_hours", "attendance_pct", "sleep_hours", "previous_score", "assignments_done", "has_job_num"]
X = df[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("MAE with 6 features:", round(mean_absolute_error(y_test, predictions), 2))

The average miss dropped to about **5 points**. Now the model has one coefficient per feature.

🤔 **Predict:** which feature will have a **negative** coefficient?

In [ ]:
for name, coef in zip(features, model.coef_):
    print(f"{name}: {coef:.2f}")
print("intercept:", round(model.intercept_, 1))

Read each coefficient as **"one more unit of this feature changes the predicted score by this many points,
if all the other features stay the same"**:

| Feature | Coef | In plain words |
|---------|------|----------------|
| `study_hours` | +1.63 | each extra study hour per week → about **+1.6 points** |
| `attendance_pct` | +0.24 | each extra percent of attendance → +0.24 (10% more → about +2.4 points) |
| `sleep_hours` | +1.59 | each extra hour of sleep per night → about +1.6 points |
| `previous_score` | +0.36 | each extra point last term → +0.36 points now |
| `assignments_done` | +1.13 | each extra assignment handed in → about +1.1 points |
| `has_job_num` | −4.54 | having a part-time job → about **4.5 points lower**, other things equal |

Three warnings:
- **Do not compare sizes across units.** 0.24 per *percent* is not "less important" than 1.6 per *hour*: a
  percent is a much smaller step than an hour.
- **A coefficient is a pattern, not a proof of cause.** It does not prove that taking a job *causes* lower scores.
  Students with jobs may also differ in ways the data does not record.
- The intercept (−19) alone means nothing here: no real student has 0% attendance and a previous score of 0.

## 5. A second number: R² (optional)

`model.score(X_test, y_test)` for a regression model gives **R²**: the share of the differences between students'
scores that the model explains. 1.0 = explains everything, 0 = no better than guessing the average.

In [ ]:
# same 60 test students; the line model only needs their study_hours column
print("R² one feature: ", round(line_model.score(X_test[["study_hours"]], y_test), 2))
print("R² six features:", round(model.score(X_test, y_test), 2))

Six features explain about two thirds of the differences in final scores; the rest is things we did not measure
(and luck on exam day). **MAE is the number to report to non-experts:** "on average we are about 5 points off".

## 6. Predicted vs actual: a picture of the errors

A perfect model would put every test student on the red diagonal (predicted = actual). The vertical distance
from the diagonal is the miss.

In [ ]:
plt.scatter(y_test, predictions)
plt.plot([20, 100], [20, 100], color="red", label="perfect prediction")
plt.xlabel("actual final score")
plt.ylabel("predicted final score")
plt.title("Test students: most predictions within about 5 points")
plt.legend()
plt.show()

🤔 Look at the plot: does the model make bigger mistakes for low scores or high scores? (Look for points far
from the line.) This is how you spot groups of students the model serves badly.

## 7. Using the model: Asha's predicted score

In [ ]:
asha = pd.DataFrame([{
    "study_hours": 12, "attendance_pct": 90, "sleep_hours": 7.5,
    "previous_score": 70, "assignments_done": 9, "has_job_num": 0,
}])
print("Asha's predicted score:", round(model.predict(asha)[0], 1))

`model.predict(...)` returns an array (one value per row), so `[0]` takes Asha's value. A fair way to tell
Asha: *"about 70, give or take 5 points"*, because MAE ≈ 5.

## 8. Common mistakes (these cells fail **on purpose**)

In [ ]:
# Mistake 1: a text column in X. Models only accept numbers
LinearRegression().fit(df[["study_hours", "has_job"]], y)

In [ ]:
# Mistake 2: the new example is missing a feature the model was trained with
model.predict(pd.DataFrame([{"study_hours": 12, "attendance_pct": 90}]))

**Fixes:** convert text to numbers first (`has_job_num`) · give the model every training column.

**Mistake 3 (no error):** reporting the MAE on the **training** data. It is usually a bit lower (better) than on
the test set, because the model has seen those students.

In [ ]:
train_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, predictions)
print("MAE on training data:", round(train_mae, 2))
print("MAE on test data:    ", round(test_mae, 2))

Here the gap is small (a straight line cannot memorise much). Tomorrow's trees are different. Always report test.

## Recap

1. Regression predicts a **number**. `LinearRegression` finds the best straight line (one slope per feature).
2. A coefficient = "one more unit of this feature changes the prediction by this much, others fixed". Not a cause.
3. **MAE** = average distance between actual and predicted, in the target's units. Compare it with the
   baseline MAE.
4. Text features must become numbers first: `(df["has_job"] == "yes").astype(int)`.
5. A predicted-vs-actual plot shows where the errors are.

**Next time:** classification: predicting pass/fail, measuring with accuracy, and the confusion matrix.